<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

The exported checkpoints are inference graphs. RepDWConv has already been fused into one depthwise convolution, and batch-norm on the backbone is folded into the convolution bias. The recognition neck keeps its own batch-norm, which is why [`BatchNorm`](https://alix559.github.io/solid_lens/lcnet.html#batchnorm) lives next to the backbone that no longer uses it.

In [4]:
from fastcore.test import test_eq

test_eq(_unit_stride(1), True)
test_eq(_unit_stride((1, 1)), True)
test_eq(_unit_stride((2, 1)), False)
_unit_stride((2, 1))

False

A residual is legal only when the stride is 1 in both directions and the channel count does not change. The recognizer's `(2, 1)` blocks are not residuals.

## Batch-norm

In [7]:
#| echo: false
#| output: asis
show_doc(BatchNorm)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/layers/lcnet.py#L33){target="_blank" style="float:right; font-size:smaller"}

### BatchNorm

```python
def BatchNorm(
    channels:int, eps:float
)->None:
```

*Inference batch-norm. Running statistics are weights.*

This is inference batch-norm: the running statistics are weights, not buffers that update. LightSVTR uses it. The backbone does not, because those batch-norms were folded into the convolution bias before export.

## Squeeze-excite

In [9]:
#| echo: false
#| output: asis
show_doc(SqueezeExcite)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/layers/lcnet.py#L55){target="_blank" style="float:right; font-size:smaller"}

### SqueezeExcite

```python
def SqueezeExcite(
    channels:int, alpha:float, *, reduction:int=4, residual:bool=False
)->None:
```

*Global pool, 1x1 bottleneck, hardsigmoid. Optional residual adds the gated map.*

## The block

In [11]:
#| echo: false
#| output: asis
show_doc(LCNetV4Block)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/layers/lcnet.py#L83){target="_blank" style="float:right; font-size:smaller"}

### LCNetV4Block

```python
def LCNetV4Block(
    spec:Block, se_alpha:float, *, bias:bool
)->None:
```

*Token mixer (depthwise conv, optional SE) plus a 2x channel mixer.*

The residual is added to the depthwise output, not to the block input. The channel mixer is an expansion by 2, a GELU, and a projection. Submodule order here is the checkpoint order: depthwise, optional SE, expand, compress.

In [13]:
#| echo: false
#| output: asis
show_doc(stage)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/layers/lcnet.py#L116){target="_blank" style="float:right; font-size:smaller"}

### stage

```python
def stage(
    blocks:list[Block], se_alpha:float, *, bias:bool
)->Sequential:
```

## Stem

In [15]:
#| echo: false
#| output: asis
show_doc(DetStem)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/layers/lcnet.py#L123){target="_blank" style="float:right; font-size:smaller"}

### DetStem

```python
def DetStem(
    mid:int, out:int
)->None:
```

*Multi-branch stem. Total stride is 4.*

The stem's total stride is 4: one stride-2 convolution, then a stride-2 convolution after the branches merge. The 2×2 branch is padded `(0, 1, 0, 1)`, which is the asymmetric SAME the exporter used. The other branch is a 2×2 max pool with the same asymmetry. Both recognizer and detector use this stem.